# 04 - Modelling

**Author:** Juan Ruelas  
**Date:** 2026-05

We benchmark three approaches on the 2019 test set:

1. **Seasonal naive** - predict load(t) = load(t - 168h). Hard to beat for short-horizon hourly electricity, and a fair reality check on the more elaborate models.
2. **Prophet** - additive trend + multi-period seasonality + temperature as an extra regressor. Good interpretability and built-in holidays.
3. **LightGBM** - gradient-boosted trees on the full feature matrix. Handles non-linear, interacting tabular features and is the production-grade workhorse for short-term load forecasting.

All three produce hourly predictions for 2019, stored side-by-side in `predictions.csv`.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from prophet import Prophet

processed_dir = Path('..') / 'data' / 'processed'

train = pd.read_csv(processed_dir / 'features_train.csv', parse_dates=['timestamp'])
test  = pd.read_csv(processed_dir / 'features_test.csv',  parse_dates=['timestamp'])
print('train:', train.shape, ' test:', test.shape)

Importing plotly failed. Interactive plots will not work.


train: (34896, 19)  test: (8760, 19)


## 1. Seasonal naive baseline

For every test hour, predict the load observed exactly 168 hours earlier. Because our test set starts on 2019-01-01 and we have the entire 2018 in train, we can index back into the merged hourly history without any boundary issue.

This baseline already exploits the strongest signal in the data (weekly seasonality) and acts as a difficulty floor.

In [2]:
history = pd.concat([train, test], ignore_index=True).sort_values('timestamp').reset_index(drop=True)
history_lookup = history.set_index('timestamp')['load_mw']

baseline_pred = test['timestamp'].map(lambda t: history_lookup.get(t - pd.Timedelta(hours=168), np.nan))
print('Baseline NaNs:', baseline_pred.isna().sum())
baseline_pred.head()

Baseline NaNs: 0


0    40443.0
1    39414.0
2    39080.0
3    39118.0
4    39314.0
Name: timestamp, dtype: float64

## 2. Prophet with temperature regressor

Prophet expects columns named `ds` (timestamp) and `y` (target). We disable the daily seasonality default and supply our own with `Fourier order = 10` so it can capture the sharp morning/evening peaks. Temperature is added as an extra regressor - at predict time we feed the actual test-period weather (a fair operational analogue, since day-ahead weather forecasts are extremely accurate).

In [3]:
prophet_train = train.rename(columns={'timestamp': 'ds', 'load_mw': 'y'})[['ds', 'y', 'temperature_2m']]
prophet_test = test.rename(columns={'timestamp': 'ds'})[['ds', 'temperature_2m']]

model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
model_prophet.add_seasonality(name='daily', period=1, fourier_order=10)
model_prophet.add_regressor('temperature_2m')
model_prophet.fit(prophet_train)

prophet_forecast = model_prophet.predict(prophet_test)
prophet_pred = prophet_forecast['yhat'].values
prophet_pred[:5]

18:11:18 - cmdstanpy - INFO - Chain [1] start processing
18:11:29 - cmdstanpy - INFO - Chain [1] done processing


array([45180.84847839, 44875.14714803, 45434.47499244, 47280.27325569,
       51483.96623623])

## 3. LightGBM

We treat this as a standard tabular regression. The last 20% of the (already chronologically ordered) train set acts as a validation fold for early stopping - this keeps the time order intact and avoids cross-validation leakage from lag features.

Hyperparameter choices:

- `objective='regression'` with L2 loss aligns with our RMSE evaluation.
- `num_leaves=64` and `learning_rate=0.05` are a balanced starting point; early stopping decides the actual number of trees.
- `min_data_in_leaf=50` discourages over-fitting to small calendar interactions.

In [4]:
feature_cols = [c for c in train.columns if c not in ('timestamp', 'load_mw')]

n_train = len(train)
split_idx = int(n_train * 0.8)
X_tr, y_tr = train.iloc[:split_idx][feature_cols], train.iloc[:split_idx]['load_mw']
X_val, y_val = train.iloc[split_idx:][feature_cols], train.iloc[split_idx:]['load_mw']
X_te = test[feature_cols]

print('Train fold:', X_tr.shape, ' Val fold:', X_val.shape, ' Test:', X_te.shape)

Train fold: (27916, 17)  Val fold: (6980, 17)  Test: (8760, 17)


In [5]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'verbose': -1,
}

dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

model_lgbm = lgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=200)],
)

lgbm_pred = model_lgbm.predict(X_te, num_iteration=model_lgbm.best_iteration)
print('Best iteration:', model_lgbm.best_iteration)

Training until validation scores don't improve for 100 rounds
[200]	train's rmse: 498.339	val's rmse: 651.678
[400]	train's rmse: 416.194	val's rmse: 614.176
[600]	train's rmse: 368.712	val's rmse: 604.663
[800]	train's rmse: 333.003	val's rmse: 599.535
[1000]	train's rmse: 303.977	val's rmse: 597.565
[1200]	train's rmse: 279.526	val's rmse: 596.663
[1400]	train's rmse: 257.766	val's rmse: 596.237
[1600]	train's rmse: 238.677	val's rmse: 596.071
Early stopping, best iteration is:
[1548]	train's rmse: 243.501	val's rmse: 595.945
Best iteration: 1548


## 4. Persist predictions

One tidy CSV with the test timestamp, the actual load, and one column per model. Notebook 05 only needs this file.

In [6]:
preds = pd.DataFrame({
    'timestamp': test['timestamp'].values,
    'actual':    test['load_mw'].values,
    'baseline':  baseline_pred.values,
    'prophet':   prophet_pred,
    'lgbm':      lgbm_pred,
})
preds.to_csv(processed_dir / 'predictions.csv', index=False)
print('saved predictions to', processed_dir / 'predictions.csv')
preds.head()

saved predictions to ..\data\processed\predictions.csv


,timestamp,actual,baseline,prophet,lgbm
0,2019-01-01 00:00:00,41562.0,40443.0,45180.848478,41044.895108
1,2019-01-01 01:00:00,40100.0,39414.0,44875.147148,40711.231324
2,2019-01-01 02:00:00,38883.0,39080.0,45434.474992,39902.897992
3,2019-01-01 03:00:00,38806.0,39118.0,47280.273256,38879.013345
4,2019-01-01 04:00:00,38593.0,39314.0,51483.966236,38727.163697


We also persist the LightGBM booster so notebook 05 can reuse it for residual analysis and feature-importance plots without retraining.

In [7]:
model_lgbm.save_model(str(processed_dir / 'lgbm_model.txt'))
print('saved lgbm model.')

saved lgbm model.
